In [1]:
from io import BytesIO
from pathlib import Path
import tempfile
import threading
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
import xmltodict
import folium
import m3u8
import imageio_ffmpeg
from PIL import Image, ImageDraw
from ultralytics import YOLO
from tqdm.auto import tqdm
from IPython.display import display, Video
import traffic_utils as tutils

HEADERS = {"User-Agent": "Mozilla/5.0", "Referer": "https://511.vdot.virginia.gov/"}
DATA_DIR = Path("~/scr10/trafficcheck_outputs").expanduser()

In [54]:
trials = DATA_DIR.glob('*')
trial = next(iter(trials))

# Video cropping parameters
LINE_1_LEFT_Y = 0.20
LINE_1_RIGHT_Y = 0.45
LINE_2_LEFT_Y = 0.50
LINE_2_RIGHT_Y = 0.25
CROP_LINES = [LINE_1_LEFT_Y, LINE_1_RIGHT_Y, LINE_2_LEFT_Y, LINE_2_RIGHT_Y]

def read_segment(path):
    reader = imageio_ffmpeg.read_frames(str(path), pix_fmt="rgb24")
    try:
        metadata = next(reader)
        width, height = metadata["size"]
        frames = [np.frombuffer(raw, np.uint8).reshape(height, width, 3) for raw in reader]
    finally:
        reader.close()
        
    return frames, float(metadata["fps"])

# Load videos using ffmpeg
path = next(iter(trial.glob('camera_*_raw_segments')))
video_frames = []
for seg in tqdm(sorted(path.glob('segment_*.ts'))):
    frames, fps = read_segment(seg)
    video_frames.extend(frames)

mp4_path = trial / "camera_full.mp4"
tutils.write_clip(video_frames, mp4_path, fps)
display(Video(filename=str(mp4_path), embed=True, width=640))

  0%|          | 0/12 [00:00<?, ?it/s]

In [56]:
CONFIDENCE = 0.05
model = YOLO("yolo11s.pt")
vehicle_ids = [class_id for class_id, name in model.names.items() if name in {"car", "truck"}]
if not vehicle_ids:
    raise ValueError("The model has no car or truck classes.")

annotated_frames = []
sample_counts = []
for result in tqdm(tutils.detect_vehicles(video_frames, model, vehicle_ids), total=len(video_frames), desc="Sample YOLO"):
    sample_counts.append(len(result.boxes))

    frame = result.plot()[:, :, ::-1]
    annotated_frames.append(frame)
    
yolo_path = trial / "camera_full_yolo.mp4"
tutils.write_clip(annotated_frames, yolo_path, fps)
print(f"Sample mean: {np.mean(sample_counts):.2f} car/truck boxes per frame")
display(Video(filename=str(yolo_path), embed=True, width=640))

Sample YOLO:   0%|          | 0/912 [00:00<?, ?it/s]

Sample mean: 6.98 car/truck boxes per frame


In [63]:
# Sparse Optical flow
import cv2 as cv

cap = cv.VideoCapture(mp4_path)

# params for ShiTomasi corner detection
feature_params = dict( maxCorners = 100,
                       qualityLevel = 0.15,
                       minDistance = 10,
                       blockSize = 3)
 
# Parameters for lucas kanade optical flow
lk_params = dict( winSize  = (15, 15),
                  maxLevel = 2,
                  criteria = (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 0.03))

# For tracking and detecting new features
saved_tracks = []
curr_tracks = []
frame_idx = 0
detect_interval = 5

video_frames = []

def above_dist_threshold(tr, movement_to_save=15):
    tr = np.array(tr)
    diff = np.diff(tr, axis=0)
    dist = np.sum(np.linalg.norm(diff, axis=1))
    return dist > movement_to_save

while True:
    ret, frame = cap.read()
    if not ret:
        break
    vis = frame.copy()
 
    frame_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    
    if len(curr_tracks) > 0:
        img0, img1 = prev_gray, frame_gray

        # Last position of all tracked objects
        p0 = np.float32([tr[-1] for tr in curr_tracks]).reshape([-1, 1, 2])

        # Compute forward optical flow
        p1, st, err = cv.calcOpticalFlowPyrLK(img0, img1, p0, None, **lk_params)

        # Filter by elements whose reverse flow recovers original position
        p0r, st, err = cv.calcOpticalFlowPyrLK(img1, img0, p1, None, **lk_params)
        d = abs(p0 - p0r).reshape(-1, 2).max(-1)
        good = d < 1

        # Compute new tracks in the frame
        new_tracks = []
        for tr, (x, y), good_flag in zip(curr_tracks, p1.reshape([-1, 2]), good):
            if not good_flag:
                # If the object has moved more than N pixels, save it
                if above_dist_threshold(tr):
                    saved_tracks.append(np.array(tr))
                continue
            tr.append((x, y))
            new_tracks.append(tr)
            
        curr_tracks = new_tracks

    # Write all saved tracks to video frame
    # for tr in saved_tracks:
    #     cv.circle(vis, (int(tr[-1][0]), int(tr[-1][1])), 2, (255, 0, 0))
    # cv.polylines(vis, [np.int32(tr) for tr in saved_tracks], False, (255, 0, 0))

    # Write all current tracks to video frame
    plot_tracks = [tr for tr in curr_tracks if above_dist_threshold(tr)]
    for tr in plot_tracks:
        cv.circle(vis, (int(tr[-1][0]), int(tr[-1][1])), 2, (0, 255, 0))
    cv.polylines(vis, [np.int32(tr) for tr in plot_tracks], False, (0, 255, 0))   

    video_frames.append(vis)

    # Look for new features that have entered the frame
    if frame_idx % detect_interval == 0:
        mask = np.ones_like(frame_gray) * 255
        mask = tutils.crop_below_lines(mask, *CROP_LINES)
        for x, y in [np.int32(tr[-1]) for tr in curr_tracks]:
            # Don't look at existing track locations
            cv.circle(mask, (x, y), 5, 0, -1)

        p = cv.goodFeaturesToTrack(frame_gray, mask=mask, **feature_params)
        if p is not None:
            for x, y in np.float32(p).reshape(-1, 2):
                curr_tracks.append([(x, y)])

    frame_idx += 1
    prev_gray = frame_gray

optical_flow_path = trial / "sparse_optical_flow.mp4"
tutils.write_clip(video_frames, optical_flow_path, fps)
display(Video(filename=str(optical_flow_path), embed=True, width=640))

In [91]:
# Dense Optical flow
import cv2 as cv

cap = cv.VideoCapture(mp4_path)
video_frames = []

# Get first frame
ret, frame0 = cap.read()
frame0_gray = cv.cvtColor(frame0, cv.COLOR_BGR2GRAY)

# For plotting


flow_frames = []
while True:
    ret, frame1 = cap.read()
    if not ret:
        break

    frame1_gray = cv.cvtColor(frame1, cv.COLOR_BGR2GRAY)
    flow = cv.calcOpticalFlowFarneback(frame0_gray, frame1_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    flow = tutils.crop_below_lines(flow, *CROP_LINES)

    flow_frames.append(flow)
    frame0_gray = frame1_gray

# Apply temporal median filter
flow_frames = np.array(flow_frames)
flow_frames = median_filter(flow_frames, size=5, axes=0)

# Visualize
cap = cv.VideoCapture(mp4_path)
video_frames = []

hsv = np.zeros_like(frame0)
hsv[..., 1] = 255
for flow in flow_frames:
    mag, ang = cv.cartToPolar(flow[..., 0], flow[..., 1])
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 2] = cv.normalize(mag, None, 0, 255, cv.NORM_MINMAX)
    bgr = cv.cvtColor(hsv, cv.COLOR_HSV2BGR)
    video_frames.append(bgr)

optical_flow_path = trial / "dense_optical_flow.mp4"
tutils.write_clip(video_frames, optical_flow_path, fps)
display(Video(filename=str(optical_flow_path), embed=True, width=640))

In [ ]:
# Dense Optical flow with K-means clustering
import cv2 as cv
from skimage.measure import block_reduce
from scipy.ndimage import median_filter

# Visualize
cap = cv.VideoCapture(mp4_path)
idx = 0
video_frames = []
while True:
    ret, frame = cap.read()
    vis = frame.copy()

    # Find out how many clusters of flow there are
    

    flow_x = block_reduce(flow_frames[idx, ..., 0], block_size=block_size, func=np.mean)
    flow_y = block_reduce(flow_frames[idx, ..., 1], block_size=block_size, func=np.mean)
    mask = np.sqrt(flow_x**2 + flow_y**2) > threshold
    for xx, yy, ux, uy in zip(x[mask], y[mask], flow_x[mask], flow_y[mask]):
        start = np.int32((xx, yy))
        end = np.int32((xx + ux, yy + uy))
        cv.arrowedLine(vis, start, end, (0, 255, 0), 2, tipLength=0.3)
    
    video_frames.append(vis)


    idx += 1
    if idx >= 60:
        break


optical_flow_path = trial / "dense_optical_flow_vectors.mp4"
tutils.write_clip(video_frames, optical_flow_path, fps)
display(Video(filename=str(optical_flow_path), embed=True, width=640))

In [89]:
# Dense Optical flow
import cv2 as cv
from skimage.measure import block_reduce
from scipy.ndimage import median_filter

cap = cv.VideoCapture(mp4_path)
video_frames = []

# Get first frame
ret, frame0 = cap.read()
frame0_gray = cv.cvtColor(frame0, cv.COLOR_BGR2GRAY)

# For plotting
y, x = np.mgrid[:frame0_gray.shape[0], :frame0_gray.shape[1]]
block_size=(8, 8)
x = block_reduce(x, block_size=block_size, func=np.mean)
y = block_reduce(y, block_size=block_size, func=np.mean)

threshold=1

flow_frames = []
idx = 0
while True:
    ret, frame1 = cap.read()
    if not ret:
        break

    frame1_gray = cv.cvtColor(frame1, cv.COLOR_BGR2GRAY)
    flow = cv.calcOpticalFlowFarneback(frame0_gray, frame1_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    flow = tutils.crop_below_lines(flow, *CROP_LINES)

    flow_frames.append(flow)
    frame0_gray = frame1_gray

    idx += 1
    if idx >= 60:
        break

# Apply temporal median filter
flow_frames = np.array(flow_frames)
flow_frames = median_filter(flow_frames, size=5, axes=0)

# Visualize
cap = cv.VideoCapture(mp4_path)
idx = 0
video_frames = []
while True:
    ret, frame = cap.read()
    vis = frame.copy()

    flow_x = block_reduce(flow_frames[idx, ..., 0], block_size=block_size, func=np.mean)
    flow_y = block_reduce(flow_frames[idx, ..., 1], block_size=block_size, func=np.mean)
    mask = np.sqrt(flow_x**2 + flow_y**2) > threshold
    for xx, yy, ux, uy in zip(x[mask], y[mask], flow_x[mask], flow_y[mask]):
        start = np.int32((xx, yy))
        end = np.int32((xx + ux, yy + uy))
        cv.arrowedLine(vis, start, end, (0, 255, 0), 2, tipLength=0.3)
    
    video_frames.append(vis)


    idx += 1
    if idx >= 60:
        break


optical_flow_path = trial / "dense_optical_flow_vectors.mp4"
tutils.write_clip(video_frames, optical_flow_path, fps)
display(Video(filename=str(optical_flow_path), embed=True, width=640))